In [11]:
#export
"""
This is for all short and random quality-of-life utilities."""
import k1lib as k1; import k1lib.cli as cli
import flask, time, inspect, threading; from flask import request, session
__all__ = ["db", "SuccessException", "FailureException", "ShortCircuit", "app"]

In [19]:
#export
k1.settings.add("web", k1.Settings()
    .add("secretKey", "e0Q5nrFVrRy7OCGAEVZoLU40xegB6s", "Flask secret key to store session data", sensitive=True, env="K1WEB_SECRET_KEY")
    .add("speedBench", False, "whether to track each endpoint's timing data in its separate sqlite database file"));
_db = [None]
def db():
    """Grab sqlite db object used in all of k1lib.web"""
    if _db[0] is None: _db[0] = cli.sql("k1_web.db", mode="lite")["default"]
    return _db[0]
_errorsDb = [None]
def errorsDb():
    if _errorsDb[0] is None: s = db(); s.query("CREATE TABLE IF NOT EXISTS errors ( id INTEGER primary key autoincrement, time BIGINT, userId, funcName, errType, err, tb );"); _errorsDb[0] = s["errors"]
    return _errorsDb[0]
class SuccessException(Exception): pass
class FailureException(Exception): pass
class ShortCircuit(Exception): pass

In [38]:
#export
sessionAutoInc = k1.AutoIncrement(prefix=f"_sess_{round(time.time()*1000)}_"); fnameAutoInc = k1.AutoIncrement(); threadsD = {}
class Flask:
    def __init__(self, import_name, cors=False, **kwargs):
        """Initializes a new Flask instance, but with quality of life features

:param cors: if True, enable cross origin resource sharing
:param kwargs: other kwargs passed into :class:`~flask.Flask`"""
        self.import_name = import_name; self.cors = cors; self.registeredFuncs = {}; self._redis = None
        self._app = flask.Flask(import_name, **kwargs); self._app.secret_key = k1.settings.web.secretKey; self._funcNames = set()
        self._app.config["PERMANENT_SESSION_LIFETIME"] = 7*24*3600; self.routes = []
        if cors: import flask_cors; flask_cors.CORS(self._app)
        self._cdb = None
        def defaultHandler(obj):
            if "redirect" in obj: return f"<script>window.location = {json.dumps(obj['redirect'])};</script>"
            return "<script>window.location = '/';</script>"
        @self.route("/api/login", methods=["POST"])
        def login():
            js = request.json(); user = k1.web.user.login(js["username"], js["password"])
            if user: session["userId"] = user.id; return "", 302, {"Location": "/"}
            else: raise FailureException("User doesn't exist or wrong password")
        @self.route("/api/logout")
        def logout(): session["userId"] = None; return "", 302, {"Location": "/"}
    def __getattr__(self, attr): return self.__dict__[attr] if attr in self.__dict__ else getattr(self._app, attr)
    def __call__(self, *args, **kwargs): self._app(*args, **kwargs)
    def __iter__(self): return self._app.__iter__()
    def __getitem__(self, *args, **kwargs): return self._app.__getitem__(*args, **kwargs)
    def route(self, path, guard=None, raw=False, sse=False, docs="", speedBench=None, *args, **kwargs):
        """
:param guard: optional guard function, will be executed before the function, to make sure the user has logged in
:param raw: if True, bypass all sessions/userId mechanisms. Set to true for endpoints intended for bots/sensors
:param sse: whether to enable Server Side Events. If True, expects the function to be a generator
:param docs: optional docs
:param benchmark: if True, record throughput and execution time of this route"""
        def inner(f):
            if f.__name__ in self._funcNames: print(f"Warning: @app.route encountered function '{f.__name__}' (file '{inspect.getfile(f)}'), which has appeared before")
            self._funcNames.add(f.__name__); ogF = f; nicePath = "route_" + path.replace(*"/-").replace(*"<_").replace(*">_").replace(*":_")
            _speedBench = k1.settings.web.speedBench if speedBench is None else speedBench
            if not sse and _speedBench: f = k1.speed(name=nicePath, fn=f"speeds/{nicePath}", docs="Automated app.route() benchmark")(f)
            self.routes.append({"route": path, "func": ogF, "docs": docs, "args": args, "kwargs": kwargs, "hasGuard": guard is not None})
            def tryf(*args, **kwargs):
                currentThreadId = id(threading.current_thread()); threadsD[currentThreadId] = {"request.path": request.path, "path": path, "time": time.time()}
                durationW = k1.Wrapper([time.time(), None])
                def cleanup(): del threadsD[currentThreadId]; durationW.value[1] = time.time()
                try:
                    guardRes = guard() if guard else None; session.permanent = True; kw = kwargs
                    if "guardRes" in inspect.getfullargspec(ogF).args: kw = {"guardRes": guardRes, **kw} # if guardRes is in the arguments of the function, then inject it in
                    if "user" in inspect.getfullargspec(ogF).args: kw = {"user": k1.web.user.get(session.get("userId", None)), **kw}
                    res = flask.Response(f(*args, **kw), mimetype='text/event-stream') if sse else f(*args, **kw)
                    cleanup(); return res
                except SuccessException as e: cleanup(); return json.dumps({"msg": f"{e}"}), 202, {"Content-Type": "application/json"}
                except FailureException as e: cleanup(); return json.dumps({"exc": f"{e}", "tb": f"{traceback.format_exc()}"}), 418, {"Content-Type": "application/json"}
                except ShortCircuit as e: cleanup(); return e.res
                except Exception as e:
                    errorsDb().insert(time=int(time.time()), userId=session.get("userId"), funcName=f"{f.__name__}", errType=f"{type(e)}", err=f"{e}", tb=f"{traceback.format_exc()}")
                    cleanup(); return json.dumps({"exc": f"{e}"}), 418, {"Content-Type": "application/json"}
            tryf.__name__ = f"{f.__name__}{fnameAutoInc()}"; self._app.route(path, *args, **kwargs)(tryf); return ogF
        return inner
    def register(self, dummy=None):
        """Registers this function to the system. Other mechanisms can access any functions registered here. Kinda like a custom :meth:`route`, but I can inject in my logic instead."""
        def inner(f):
            def tryf(*args, **kwargs):
                try: return f(*args, **kwargs) # intentionally don't have SuccessException and FailureException because this is supposed to be an internal function, so anything that wants to display to the user is considered to be problematic
                except ShortCircuit as e: return e.res
                except Exception as e: errorsDb().insert(time=int(time.time()), userId=session.get("userId"), funcName=f"{f.__name__}", errType=f"{type(e)}", err=f"{e}", tb=f"{traceback.format_exc()}")
            tryf.__name__ = f.__name__; self.registeredFuncs[f.__name__] = tryf
        return inner
    def flask(self, **kwargs): # sets up /k1/docs for all endpoint docs
        @self.route("/k1/docs", **kwargs)
        def docs(): return self.routes | cli.apply(lambda x: [f"<div style='color: {'green' if x['hasGuard'] else 'red'}'>{html.escape(repr(x['route']))}</div>", x["func"].__name__, x["args"], x["kwargs"], inspect.getfile(x["func"]), x["func"].__doc__, x["docs"]]) | cli.apply(cli.aS(repr) | cli.aS(html.escape), [1,2,3,4,5,6]) | cli.sort(0, False) | cli.deref() | (cli.toJsFunc("term") | cli.grep("${term}") | k1.viz.Table(["route", "funcName", "args", "kwargs", "fileName", "docstring", "explicit docs"], height=600, sortF=True)) | cli.op().interface() | cli.toHtml()
_app = [None]
def app(import_name):
    if _app[0] is None: _app[0] = Flask(import_name)
    return _app[0]

In [40]:
app()

In [1]:
import k1lib.web as web

In [4]:
web.init.app()

In [1]:
from k1lib.imports import *

In [2]:
s = sql("a.db", mode="lite")["default"]

In [4]:
s.query("""
CREATE TABLE users (
    id       INTEGER primary key autoincrement,
    name     VARCHAR(50),
    age      INT,
    time     BIGINT,
    groupIds BIGINT[],
    data     JSON
);""")

[]

In [6]:
s["users"].insert(name="abc", age=3, time=time.time(), groupIds=[3, 4, 5], data={"a": 3})

(id=1, name=(3 len) "abc", age=3, time=1744493727.3143885, groupIds=[3, 4, 5], data={'a': 3})

In [12]:
s["users"] | ls()

Table `users` (#rows(approx):1, len(row):6)

id   name    age   time                 groupIds      data         
1    'abc'   3     1744493727.3143885   '[3, 4, 5]'   '{"b": 4}'   
...
...
id   name    age   time                 groupIds      data         
1    'abc'   3     1744493727.3143885   '[3, 4, 5]'   '{"b": 4}'   

Table format:
cid   name       type          notnull   dflt_value   pk   
0     id         INTEGER       0         None         1    
1     name       VARCHAR(50)   0         None         0    
2     age        INT           0         None         0    
3     time       BIGINT        0         None         0    
4     groupIds   BIGINT[]      0         None         0    
5     data       JSON          0         None         0    


In [11]:
s["users"][1].data = {"b": 4}

In [13]:
s["users"][1].data

{'b': 4}

In [14]:
!../../export.py web/init --upload=True

./export started up - /home/quang/miniforge3/bin/python
----- exportAll
17285   0   60%   
11364   1   40%   
installing...
Found existing installation: k1lib 1.8
Uninstalling k1lib-1.8:
  Successfully uninstalled k1lib-1.8
DEPRECATION: Loading egg at /home/quang/miniforge3/lib/python3.12/site-packages/aigu-0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://pypi.org/simple, http://10.104.0.3:3141/
Processing /home/quang/k1lib
  Preparing metadata (setup.py) ... done
  Created wheel for k1lib: filename=k1lib-1.8-py3-none-any.whl size=5156430 sha256=393ff92761b2f8cc10a2f1323a93b0d5b8f4ed2dca37fcbcb448fd56e5d51c2c
  Stored in directory: /tmp/pip-ephem-wheel-cache-pk5nddns/wheels/b5/32/67/e20c84dce16d707fb881c12d405f70adfaa36fe7dae9021380
Successfully built k1lib
installed
uploading...
uploaded


In [3]:
!../../export.py web/init

./export started up - /home/quang/miniforge3/bin/python
----- exportAll
17285   0   60%   
11364   1   40%   
installing...
Found existing installation: k1lib 1.8
Uninstalling k1lib-1.8:
  Successfully uninstalled k1lib-1.8
DEPRECATION: Loading egg at /home/quang/miniforge3/lib/python3.12/site-packages/aigu-0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://pypi.org/simple, http://10.104.0.3:3141/
Processing /home/quang/k1lib
  Preparing metadata (setup.py) ... done
  Created wheel for k1lib: filename=k1lib-1.8-py3-none-any.whl size=5156396 sha256=33c8fb2cbf3f9f5c7d17d549217184feaafeb19c2698f12d315f8641d1dfe5e7
  Stored in directory: /tmp/pip-ephem-wheel-cache-dwxbrqmo/wheels/b5/32/67/e20c84dce16d707fb881c12d405f70adfaa36fe7dae9021380
Successfully built k1lib
installed


In [1]:
!../../export.py web/init --bootstrap=True

./export started up - /home/quang/miniforge3/bin/python
----- bootstrapping
Current dir: /home/quang/k1lib, /home/quang/k1lib/k1lib/web/../../export.py
installing...
Found existing installation: k1lib 1.8
Uninstalling k1lib-1.8:
  Successfully uninstalled k1lib-1.8
DEPRECATION: Loading egg at /home/quang/miniforge3/lib/python3.12/site-packages/aigu-0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://pypi.org/simple, http://10.104.0.3:3141/
Processing /home/quang/k1lib
  Preparing metadata (setup.py) ... done
  Created wheel for k1lib: filename=k1lib-1.8-py3-none-any.whl size=5153276 sha256=a25c699231094d10d05037a044df7d16e800d859b72ac5ac7bb3cf10a04718d3
  Stored in directory: /tmp/pip-ephem-wheel-cache-pttdzruz/wheels/b5/32/67/e20c84dce16d707fb881c12d405f70adfaa36fe7dae9021380
Successfully built k1lib
insta